# GNN XAI 해석 - 특허 소송 링크 예측
**SimpleCatHeteroGAT + BCE Loss + Edge-split** 모델에 대한 7가지 XAI 방법 적용

| 방법 | 설명 |
|------|------|
| M1 Attention Visualization | GAT attention weight 추출 → 어떤 이웃이 임베딩에 기여했는지 |
| M2 Grad-CAM | gradient × activation → 어떤 피처/레이어가 소송 예측에 기여했는지 |
| M3 Counterfactual | 엣지 타입별 제거 후 확률 변화 측정 |
| M4 GNNExplainer | 예측 유지에 필요한 핵심 edge mask 학습 |
| M5 PGExplainer | MLP(h_i \|\| h_j) 학습 → 이웃 노드 중요도 |
| M6 SubgraphX | Monte Carlo Shapley → 핵심 서브그래프 노드 |
| M7 GraphSHAP | KernelSHAP → 입력 피처 그룹별 기여도 |


In [ ]:
import sys, warnings, copy
sys.path.insert(0, 'viz_app')
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from collections import defaultdict
from math import comb as math_comb

if not hasattr(np, '_core'):
    import numpy.core as _nc
    sys.modules['numpy._core'] = _nc
    sys.modules['numpy._core.multiarray'] = _nc.multiarray

from backend.model import SimpleCatHeteroGAT, EDGE_TYPES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Edge types: {len(EDGE_TYPES)}')


In [ ]:
ckpt = torch.load('phase3_simplecat_ckpt.pt', map_location='cpu', weights_only=False)
data = torch.load('hetero_data.pt', map_location='cpu', weights_only=False).to(device)

model = SimpleCatHeteroGAT(n_ipc=len(ckpt['ipc_list'])).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

pat_list   = ckpt['pat_list']
ipc_list   = ckpt['ipc_list']
co_list    = ckpt['co_list']
pat_to_idx = ckpt['pat_to_idx']
best_thr   = float(ckpt['best_threshold'])
tr_result  = ckpt['test_result']

sued_e     = np.array(ckpt['sued_edges'])
te_idx     = np.array(ckpt['test_edge_idx'])
test_edges = sued_e[:, te_idx]

with torch.no_grad():
    h_all = model(data)
h_pat = h_all['patent'].detach().cpu()

print(f"Patents: {len(pat_list)}, IPCs: {len(ipc_list)}, Companies: {len(co_list)}")
print(f"Test edges: {test_edges.shape[1]}")
print(f"Best threshold: {best_thr:.2f}")
print(f"\nTest Results:")
for k, v in tr_result.items():
    if k != 'best_epoch':
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# 각 test edge의 BCE 예측 확률 계산
test_pairs = []
h_np = h_pat.numpy()
for j in range(test_edges.shape[1]):
    s, d = int(test_edges[0, j]), int(test_edges[1, j])
    logit = float(np.dot(h_np[s], h_np[d]))
    prob  = 1.0 / (1.0 + np.exp(-logit))
    test_pairs.append((s, d, prob))

test_pairs.sort(key=lambda x: -x[2])

print("Test 엣지 예측 확률 (상위 15개):")
print(f"{'#':<4} {'원고':^22} {'피고':^22} {'P(sue)':>8} {'예측':>5}")
print('-'*70)
for k_i, (s, d, p) in enumerate(test_pairs[:15]):
    flag = 'O' if p >= best_thr else ' '
    print(f"{k_i+1:<4} {str(pat_list[s])[:22]:<22} {str(pat_list[d])[:22]:<22} {p:>7.3f} {flag:>5}")

# 분석 대상: 예측 확률 가장 높은 쌍
QUERY_S, QUERY_D, QUERY_P = test_pairs[0]
print(f"\n분석 대상:")
print(f"  원고: {pat_list[QUERY_S]} (idx={QUERY_S})")
print(f"  피고: {pat_list[QUERY_D]} (idx={QUERY_D})")
print(f"  예측 확률: {QUERY_P:.4f}  (threshold: {best_thr:.2f})")


## M1. Attention Visualization in GAT
GAT의 각 레이어에서 edge type별 attention weight를 추출합니다.
→ 어떤 이웃 노드가 임베딩 형성에 가장 많이 기여했는지 시각화합니다.


In [ ]:
def get_attention_per_layer(model, data, target_idx, target_type='patent'):
    """Step-by-step forward, collecting attention weights for target node."""
    x_pat_raw = torch.cat([data['patent'].struct, data['patent'].x], dim=1)
    x = {
        'patent':  model.pat_proj(x_pat_raw),
        'company': F.leaky_relu(model.co_proj(data['company'].x)),
        'ipc':     model.ipc_emb.weight.clone(),
    }
    acts = [F.leaky_relu, F.leaky_relu, torch.tanh]
    all_attn = {}

    for li, (hconv, bn, act) in enumerate(zip(model.convs[:3], model.bns, acts)):
        x_agg = defaultdict(list)
        layer_attn = {}

        for et in EDGE_TYPES:
            src_t, rel, dst_t = et
            if et not in data.edge_index_dict or et not in hconv.convs:
                continue
            et_conv  = hconv.convs[et]
            edge_idx = data.edge_index_dict[et]
            with torch.no_grad():
                out, (ei_ret, alpha) = et_conv(
                    (x[src_t], x[dst_t]), edge_idx,
                    return_attention_weights=True)
            if dst_t == target_type:
                mask = (ei_ret[1] == target_idx)
                if mask.any():
                    layer_attn[et] = {
                        'src': ei_ret[0][mask].cpu().tolist(),
                        'attn': alpha[mask].mean(dim=1).cpu().tolist()
                    }
            x_agg[dst_t].append(out)

        all_attn[f'L{li+1}'] = layer_attn
        x_new = {k: torch.stack(vs).mean(0) for k, vs in x_agg.items()}
        x = {k: model.drop(act(bn[k](v.view(-1, model.hid)))) if k in bn
             else model.drop(act(v))
             for k, v in x_new.items()}
    return all_attn


with torch.no_grad():
    attn_src = get_attention_per_layer(model, data, QUERY_S)
    attn_dst = get_attention_per_layer(model, data, QUERY_D)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f'M1. Attention Visualization\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}', fontsize=13)

for row_i, (node_idx, node_attn, node_label) in enumerate([
    (QUERY_S, attn_src, f'원고 {pat_list[QUERY_S]}'),
    (QUERY_D, attn_dst, f'피고 {pat_list[QUERY_D]}')
]):
    for col_i, layer in enumerate(['L1', 'L2', 'L3']):
        ax = axes[row_i][col_i]
        ax.set_title(f'{node_label} | {layer}', fontsize=9)
        if layer not in node_attn or not node_attn[layer]:
            ax.text(0.5, 0.5, 'No edges', ha='center', va='center')
            ax.axis('off'); continue

        bars = []
        for et, v in node_attn[layer].items():
            src_t, rel, dst_t = et
            for sn, av in zip(v['src'], v['attn']):
                if src_t == 'patent': lbl = str(pat_list[sn])[:12]
                elif src_t == 'ipc':  lbl = ipc_list[sn] if sn < len(ipc_list) else f'IPC{sn}'
                else:                 lbl = co_list[sn][:10] if sn < len(co_list) else f'CO{sn}'
                bars.append((lbl, av, rel))
        if not bars:
            ax.text(0.5, 0.5, 'No incoming edges', ha='center', va='center')
            ax.axis('off'); continue
        bars.sort(key=lambda x: -x[1])
        bars = bars[:8]
        labels_b = [b[0] for b in bars]; vals_b = [b[1] for b in bars]
        rels_b   = [b[2] for b in bars]
        urels = list(dict.fromkeys(rels_b))
        cmap  = {r: f'C{i2}' for i2, r in enumerate(urels)}
        ax.barh(range(len(bars)), vals_b,
                color=[cmap[r] for r in rels_b], alpha=0.8)
        ax.set_yticks(range(len(bars))); ax.set_yticklabels(labels_b, fontsize=7)
        ax.set_xlabel('Attention weight')
        ax.legend(handles=[Patch(color=cmap[r], label=r) for r in urels], fontsize=6)

plt.tight_layout()
plt.savefig('xai_m1_attention.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m1_attention.png")


## M2. Grad-CAM for Link Prediction
`score = sigmoid(h_src · h_dst)` 에 대한 gradient × activation → 레이어별, 노드별 중요도


In [ ]:
def grad_cam_link(model, data, src_idx, dst_idx):
    model.eval()
    x_pat_raw = torch.cat([data['patent'].struct, data['patent'].x], dim=1)
    x = {
        'patent':  model.pat_proj(x_pat_raw).requires_grad_(True),
        'company': F.leaky_relu(model.co_proj(data['company'].x)),
        'ipc':     model.ipc_emb.weight,
    }
    acts_fns = [F.leaky_relu, F.leaky_relu, torch.tanh]
    intermediate = []

    for li, (conv, bn, act) in enumerate(zip(model.convs[:3], model.bns, acts_fns)):
        x_new  = conv(x, data.edge_index_dict)
        x_next = {k: model.drop(act(bn[k](v.view(-1, model.hid)))) if k in bn
                  else model.drop(act(v))
                  for k, v in x_new.items()}
        if 'patent' in x_next:
            x_next['patent'] = x_next['patent'].requires_grad_(True)
            intermediate.append(x_next['patent'])
        x = x_next

    x_out = model.convs[3](x, data.edge_index_dict)
    h     = {k: v.view(-1, model.out) for k, v in x_out.items()}
    score = torch.sigmoid((h['patent'][src_idx] * h['patent'][dst_idx]).sum())
    score.backward()

    results = {}
    for li, act_t in enumerate(intermediate):
        if act_t.grad is None: continue
        for ni, tag in [(src_idx, 'src'), (dst_idx, 'dst')]:
            g = act_t.grad[ni].detach().cpu()
            a = act_t[ni].detach().cpu()
            cam = F.relu(g * a).numpy()
            results.setdefault(f'Layer {li+1}', {})[tag] = cam
    return results, score.item()

grad_res, pred_score = grad_cam_link(model, data, QUERY_S, QUERY_D)
n_layers = len(grad_res)

fig, axes = plt.subplots(2, n_layers, figsize=(5*n_layers, 8))
fig.suptitle(
    f'M2. Grad-CAM: Feature Importance per Layer\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}  |  P(sue)={pred_score:.3f}',
    fontsize=12)

for col_i, (layer_name, vals) in enumerate(grad_res.items()):
    for row_i, (tag, node_label) in enumerate([
        ('src', f'원고 {pat_list[QUERY_S]}'),
        ('dst', f'피고 {pat_list[QUERY_D]}')
    ]):
        ax = axes[row_i][col_i]
        cam = vals.get(tag, np.zeros(1))
        top_idx = np.argsort(-cam)[:20]
        ax.bar(range(min(20, len(top_idx))), cam[top_idx[:20]],
               color='steelblue', alpha=0.8)
        ax.set_title(f'{layer_name}\n{node_label}', fontsize=8)
        ax.set_xlabel('Feature dim (top-20)')
        ax.set_ylabel('Grad-CAM score')

plt.tight_layout()
plt.savefig('xai_m2_gradcam.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m2_gradcam.png")

print("\n레이어별 총 Grad-CAM 중요도:")
for layer_name, vals in grad_res.items():
    for tag in ['src', 'dst']:
        print(f"  {layer_name} {'원고' if tag=='src' else '피고'}: {vals.get(tag, np.array([0])).sum():.4f}")


## M3. Counterfactual Explanations
각 edge type의 엣지를 제거했을 때 소송 예측 확률이 얼마나 변하는지 측정합니다.
→ **확률 감소가 클수록 그 edge type이 예측에 중요**합니다.


In [ ]:
def counterfactual_by_edge_type(model, data, src_idx, dst_idx):
    with torch.no_grad():
        h0 = model(data)['patent']
        orig_prob = float(torch.sigmoid((h0[src_idx] * h0[dst_idx]).sum()))

    results = {'_Original': {'prob': orig_prob, 'n_removed': 0, 'delta': 0.0}}

    for et in EDGE_TYPES:
        src_t, rel, dst_t = et
        if et not in data.edge_index_dict: continue
        orig_edge = data.edge_index_dict[et].clone()
        keep = torch.ones(orig_edge.shape[1], dtype=torch.bool)
        if src_t == 'patent':
            keep &= (orig_edge[0] != src_idx) & (orig_edge[0] != dst_idx)
        if dst_t == 'patent':
            keep &= (orig_edge[1] != src_idx) & (orig_edge[1] != dst_idx)
        n_rem = int((~keep).sum())
        if n_rem == 0: continue

        data.edge_index_dict[et] = orig_edge[:, keep]
        with torch.no_grad():
            h_cf = model(data)['patent']
            cf_prob = float(torch.sigmoid((h_cf[src_idx] * h_cf[dst_idx]).sum()))
        data.edge_index_dict[et] = orig_edge

        results[f'{src_t[:3]}--{rel}--{dst_t[:3]}'] = {
            'prob': cf_prob, 'n_removed': n_rem, 'delta': orig_prob - cf_prob
        }
    return results

cf_res   = counterfactual_by_edge_type(model, data, QUERY_S, QUERY_D)
orig_prob = cf_res['_Original']['prob']
items_cf  = [(k, v) for k, v in cf_res.items() if k != '_Original']
items_cf.sort(key=lambda x: -abs(x[1]['delta']))

labels_cf = [x[0] for x in items_cf]
deltas_cf = [x[1]['delta'] for x in items_cf]
probs_cf  = [x[1]['prob']  for x in items_cf]
nrem_cf   = [x[1]['n_removed'] for x in items_cf]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'M3. Counterfactual: 엣지 제거시 소송 확률 변화\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}  |  원래 P={orig_prob:.3f}',
    fontsize=12)

ax1.barh(range(len(labels_cf)), deltas_cf,
         color=['red' if d > 0 else 'steelblue' for d in deltas_cf], alpha=0.8)
ax1.axvline(0, color='black', linewidth=1)
ax1.set_yticks(range(len(labels_cf))); ax1.set_yticklabels(labels_cf, fontsize=9)
ax1.set_xlabel('Δ 확률 (원래 − 제거후)\n양수 = 이 edge type이 중요함')
ax1.set_title('Edge Type별 중요도')

ax2.barh(range(len(labels_cf)), probs_cf, alpha=0.8, color='steelblue')
ax2.axvline(orig_prob, color='red', linestyle='--', label=f'원래 P ({orig_prob:.3f})')
ax2.set_yticks(range(len(labels_cf)))
ax2.set_yticklabels([f'{l} (제거 {n}개)' for l, n in zip(labels_cf, nrem_cf)], fontsize=8)
ax2.set_xlabel('소송 예측 확률'); ax2.set_title('제거 후 확률'); ax2.legend()

plt.tight_layout()
plt.savefig('xai_m3_counterfactual.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m3_counterfactual.png")

print(f"\n{'Edge Type':<35} {'제거수':>6} {'원래P':>7} {'CF_P':>7} {'Δ':>7}")
print('-'*65)
for lbl, v in items_cf:
    print(f"{lbl:<35} {v['n_removed']:>6} {orig_prob:>7.3f} {v['prob']:>7.3f} {v['delta']:>+7.3f}")


## M4. GNNExplainer (Link Prediction 버전)
예측 확률을 최대화하면서 엣지 마스크를 학습합니다.
→ **mask prob이 높은 엣지일수록 예측에 필수적인 연결**입니다.


In [ ]:
class LinkGNNExplainer:
    """GNNExplainer for heterogeneous link prediction. Learns edge masks."""
    def __init__(self, model, data, lr=0.05, n_epochs=300, coeff_size=5e-4):
        self.model = model; self.data = data
        self.lr = lr; self.n_epochs = n_epochs; self.coeff = coeff_size

    def explain(self, src_idx, dst_idx):
        local_masks = {}
        params = []
        orig_edges = {}

        for et in EDGE_TYPES:
            src_t, rel, dst_t = et
            if et not in self.data.edge_index_dict: continue
            eidx = self.data.edge_index_dict[et]
            orig_edges[et] = eidx.clone()
            rel_mask = torch.zeros(eidx.shape[1], dtype=torch.bool)
            if src_t == 'patent':
                rel_mask |= (eidx[0] == src_idx) | (eidx[0] == dst_idx)
            if dst_t == 'patent':
                rel_mask |= (eidx[1] == src_idx) | (eidx[1] == dst_idx)
            n_rel = int(rel_mask.sum())
            if n_rel == 0: continue
            mask_logit = nn.Parameter(torch.zeros(n_rel))
            params.append(mask_logit)
            local_masks[et] = (rel_mask, mask_logit)

        if not params: return {}, 0.0
        optimizer = torch.optim.Adam(params, lr=self.lr)

        for epoch in range(self.n_epochs):
            optimizer.zero_grad()
            for et, (rel_mask, mask_logit) in local_masks.items():
                eidx = orig_edges[et]
                soft  = torch.sigmoid(mask_logit)
                hard  = (torch.rand_like(soft) < soft.detach()).float()
                kept  = hard.bool()
                full_keep = torch.ones(eidx.shape[1], dtype=torch.bool)
                full_keep[rel_mask] = kept
                self.data.edge_index_dict[et] = eidx[:, full_keep]

            h_m   = self.model(self.data)['patent']
            score = torch.sigmoid((h_m[src_idx] * h_m[dst_idx]).sum())
            for et in local_masks: self.data.edge_index_dict[et] = orig_edges[et]

            size_loss = sum(torch.sigmoid(ml).mean() for _, (_, ml) in local_masks.items())
            loss = -torch.log(score + 1e-6) + self.coeff * size_loss
            loss.backward()
            optimizer.step()

        results = {}
        for et, (rel_mask, mask_logit) in local_masks.items():
            eidx = orig_edges[et]; src_t, rel_name, dst_t = et
            mp = torch.sigmoid(mask_logit).detach().cpu().numpy()
            results[et] = {
                'src_nodes': eidx[0][rel_mask].cpu().numpy(),
                'dst_nodes': eidx[1][rel_mask].cpu().numpy(),
                'mask_probs': mp, 'rel': rel_name,
                'src_t': src_t, 'dst_t': dst_t
            }
        return results

print("GNNExplainer 학습 중 (약 10-30초)...")
gnn_exp = LinkGNNExplainer(model, data, lr=0.05, n_epochs=300)
gnn_res = gnn_exp.explain(QUERY_S, QUERY_D)
print(f"완료. {len(gnn_res)}개 edge type에서 mask 학습.")

if gnn_res:
    n_et = len(gnn_res)
    fig, axes = plt.subplots(1, n_et, figsize=(max(6, 5*n_et), 6))
    if n_et == 1: axes = [axes]
    fig.suptitle(
        f'M4. GNNExplainer: 예측에 핵심적인 엣지 (mask prob)\n'
        f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}', fontsize=12)

    for ax_i, (et, v) in enumerate(gnn_res.items()):
        ax = axes[ax_i]; src_t, dst_t = v['src_t'], v['dst_t']
        def nlbl(idx, nt):
            if nt=='patent': return str(pat_list[idx])[:10]
            if nt=='ipc':    return ipc_list[idx] if idx < len(ipc_list) else f'IPC{idx}'
            return co_list[idx][:8] if idx < len(co_list) else f'CO{idx}'
        elabels = [f'{nlbl(int(s), src_t)}→{nlbl(int(d), dst_t)}'
                   for s, d in zip(v['src_nodes'], v['dst_nodes'])]
        sidx = np.argsort(-v['mask_probs'])[:10]
        mp_top = v['mask_probs'][sidx]
        ax.barh(range(len(sidx)), mp_top,
                color=['green' if p > 0.5 else 'lightgray' for p in mp_top], alpha=0.9)
        ax.axvline(0.5, color='red', linestyle='--', alpha=0.7, label='중요도 임계(0.5)')
        ax.set_yticks(range(len(sidx)))
        ax.set_yticklabels([elabels[i2] for i2 in sidx], fontsize=7)
        ax.set_xlabel('Mask probability'); ax.set_title(v['rel'], fontsize=9)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig('xai_m4_gnnexplainer.png', dpi=120, bbox_inches='tight')
    plt.show(); print("저장: xai_m4_gnnexplainer.png")


## M5. PGExplainer
`MLP(h_i ‖ h_j) → [0,1]` 를 학습하여 이웃 노드의 중요도를 파라미터화합니다.
→ 학습 후 QUERY_S, QUERY_D의 이웃에 적용하여 중요한 이웃 특허를 찾습니다.


In [ ]:
class PGEdgeMLP(nn.Module):
    def __init__(self, emb_dim=64, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2*emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, 1))
    def forward(self, hi, hj):
        return self.net(torch.cat([hi, hj], dim=-1)).squeeze(-1)

def train_pg(h_np, sued_edges, n_epochs=300, lr=1e-3):
    h_t = torch.tensor(h_np, dtype=torch.float32)
    pg  = PGEdgeMLP(emb_dim=h_np.shape[1])
    opt = torch.optim.Adam(pg.parameters(), lr=lr)
    rng = np.random.default_rng(42)
    pos_s = torch.tensor(sued_edges[0], dtype=torch.long)
    pos_d = torch.tensor(sued_edges[1], dtype=torch.long)
    losses = []
    for ep in range(n_epochs):
        opt.zero_grad()
        ns = torch.tensor(rng.integers(0, len(h_np), size=len(pos_s)), dtype=torch.long)
        nd = torch.tensor(rng.integers(0, len(h_np), size=len(pos_s)), dtype=torch.long)
        all_s = torch.cat([pos_s, ns]); all_d = torch.cat([pos_d, nd])
        lbl   = torch.cat([torch.ones(len(pos_s)), torch.zeros(len(ns))])
        logits = pg(h_t[all_s], h_t[all_d])
        loss   = F.binary_cross_entropy_with_logits(logits, lbl)
        loss.backward(); opt.step()
        if (ep+1) % 100 == 0: losses.append(loss.item())
    return pg, losses

print("PGExplainer MLP 학습 중 (약 20초)...")
pg_mlp, pg_losses = train_pg(h_pat.numpy(), sued_e, n_epochs=300)
pg_mlp.eval()
print(f"완료. 최종 loss: {pg_losses[-1]:.4f}")

def get_patent_neighbors(data, node_idx):
    nb = set()
    for et in EDGE_TYPES:
        src_t, rel, dst_t = et
        if et not in data.edge_index_dict: continue
        eidx = data.edge_index_dict[et].cpu().numpy()
        if src_t == 'patent' and dst_t == 'patent':
            nb.update(eidx[1][eidx[0] == node_idx].tolist())
            nb.update(eidx[0][eidx[1] == node_idx].tolist())
    return nb - {node_idx}

h_t = torch.tensor(h_pat.numpy(), dtype=torch.float32)
src_nb = get_patent_neighbors(data, QUERY_S)
dst_nb = get_patent_neighbors(data, QUERY_D)

with torch.no_grad():
    src_scores = {nb: float(torch.sigmoid(pg_mlp(h_t[QUERY_S].unsqueeze(0), h_t[nb].unsqueeze(0))))
                  for nb in src_nb}
    dst_scores = {nb: float(torch.sigmoid(pg_mlp(h_t[QUERY_D].unsqueeze(0), h_t[nb].unsqueeze(0))))
                  for nb in dst_nb}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'M5. PGExplainer: 소송 예측에 기여하는 이웃 특허\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}', fontsize=12)

for ax, scores, node_lbl, bar_color in [
    (ax1, src_scores, f'원고 {str(pat_list[QUERY_S])[:15]} 이웃', 'royalblue'),
    (ax2, dst_scores, f'피고 {str(pat_list[QUERY_D])[:15]} 이웃', 'coral')
]:
    if scores:
        top = sorted(scores.items(), key=lambda x: -x[1])[:10]
        lbls = [str(pat_list[n])[:16] for n, _ in top]
        vals = [v for _, v in top]
        ax.barh(range(len(top)), vals, color=bar_color, alpha=0.8)
        ax.set_yticks(range(len(top))); ax.set_yticklabels(lbls, fontsize=8)
        ax.set_xlabel('PG Importance Score')
    ax.set_title(node_lbl, fontsize=10)

plt.tight_layout()
plt.savefig('xai_m5_pgexplainer.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m5_pgexplainer.png")


## M6. SubgraphX (Monte Carlo Shapley)
이웃 노드들의 Shapley value를 Monte Carlo 샘플링으로 추정합니다.
→ 노드를 추가할 때 예측 확률이 얼마나 올라가는지 측정합니다.


In [ ]:
def pred_with_masked_nodes(model, data, src_idx, dst_idx, masked_nodes):
    """Masked nodes의 input feature를 0으로 설정 후 예측."""
    orig_x   = data['patent'].x.clone()
    orig_str = data['patent'].struct.clone()
    for n in masked_nodes:
        data['patent'].x[n]      = 0.0
        data['patent'].struct[n] = 0.0
    with torch.no_grad():
        h = model(data)['patent']
        score = float(torch.sigmoid((h[src_idx] * h[dst_idx]).sum()))
    data['patent'].x      = orig_x
    data['patent'].struct = orig_str
    return score

def mc_shapley(model, data, src_idx, dst_idx, candidates, n_samples=80):
    rng = np.random.default_rng(42)
    shapley = defaultdict(float); counts = defaultdict(int)
    cands = list(candidates)
    for _ in range(n_samples):
        perm = rng.permutation(cands)
        S    = set()
        f_S  = pred_with_masked_nodes(model, data, src_idx, dst_idx,
                                       set(cands) - S)
        for v in perm:
            S.add(v)
            f_Sv = pred_with_masked_nodes(model, data, src_idx, dst_idx,
                                           set(cands) - S)
            shapley[v] += f_Sv - f_S; counts[v] += 1; f_S = f_Sv
    return {v: shapley[v] / counts[v] for v in cands}

candidates = list((src_nb | dst_nb) - {QUERY_S, QUERY_D})[:25]
print(f"SubgraphX: {len(candidates)}개 노드 Shapley 계산 중 (약 1-2분)...")
shap_vals = mc_shapley(model, data, QUERY_S, QUERY_D, candidates, n_samples=60)
print("완료.")

sorted_sv  = sorted(shap_vals.items(), key=lambda x: -abs(x[1]))[:15]
lbls_sv    = [str(pat_list[n])[:18] for n, _ in sorted_sv]
vals_sv    = [v for _, v in sorted_sv]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(sorted_sv)), vals_sv,
        color=['red' if v > 0 else 'steelblue' for v in vals_sv], alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(range(len(sorted_sv))); ax.set_yticklabels(lbls_sv, fontsize=9)
ax.set_xlabel('Shapley Value\n(양수=기여, 음수=방해)')
ax.set_title(
    f'M6. SubgraphX: 이웃 노드별 Shapley Value\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}', fontsize=12)
ax.legend(handles=[Patch(color='red', label='기여(양수)'), Patch(color='steelblue', label='방해(음수)')])
plt.tight_layout()
plt.savefig('xai_m6_subgraphx.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m6_subgraphx.png")


## M7. GraphSHAP (KernelSHAP-style Feature Attribution)
입력 피처를 그룹으로 나눠 각 그룹의 Shapley value를 KernelSHAP으로 추정합니다.
→ struct (3d) vs FAISS 임베딩 구간별 기여도를 확인합니다.


In [ ]:
from scipy.special import comb as sp_comb

def kernel_shap_features(model, data, src_idx, dst_idx, n_groups=15, n_samples=200):
    faiss_dim = data['patent'].x.shape[1]   # 1024
    chunk = faiss_dim // (n_groups - 1)
    groups = [('struct', None, 'struct')]
    for g in range(n_groups - 1):
        s = g * chunk; e = min(s + chunk, faiss_dim)
        groups.append((f'FAISS_{s}-{e}', (s, e), 'faiss'))
    M = len(groups)

    orig_x   = data['patent'].x[src_idx].clone()
    orig_str = data['patent'].struct[src_idx].clone()

    with torch.no_grad():
        h0 = model(data)['patent']
        f0 = float(torch.sigmoid((h0[src_idx] * h0[dst_idx]).sum()))

    rng = np.random.default_rng(42)
    X_s, Y_s, W_s = [], [], []

    for _ in range(n_samples):
        inc = rng.random(M) > 0.5
        data['patent'].x[src_idx]      = orig_x.clone()
        data['patent'].struct[src_idx]  = orig_str.clone()
        for gi, (gname, grange, gtype) in enumerate(groups):
            if not inc[gi]:
                if gtype == 'struct': data['patent'].struct[src_idx] = 0.0
                else:
                    gs, ge = grange
                    data['patent'].x[src_idx, gs:ge] = 0.0
        with torch.no_grad():
            hm = model(data)['patent']
            sc = float(torch.sigmoid((hm[src_idx] * hm[dst_idx]).sum()))
        z = int(inc.sum()); z = max(1, min(z, M-1))
        w = (M - 1) / (sp_comb(M, z) * z * (M - z) + 1e-10)
        X_s.append(inc.astype(float)); Y_s.append(sc); W_s.append(w)

    data['patent'].x[src_idx]     = orig_x
    data['patent'].struct[src_idx] = orig_str

    X = np.array(X_s); Y = np.array(Y_s); W = np.array(W_s)
    Wm = np.diag(W)
    A  = X.T @ Wm @ X + 1e-8 * np.eye(M)
    b  = X.T @ Wm @ Y
    phi = np.linalg.solve(A, b)
    return {groups[gi][0]: float(phi[gi]) for gi in range(M)}

print("GraphSHAP: 원고 피처 기여도 계산 중 (약 1분)...")
shap_src = kernel_shap_features(model, data, QUERY_S, QUERY_D, n_groups=15, n_samples=200)
print("GraphSHAP: 피고 피처 기여도 계산 중...")
shap_dst = kernel_shap_features(model, data, QUERY_D, QUERY_S, n_groups=15, n_samples=200)
print("완료.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    f'M7. GraphSHAP: 입력 피처 그룹별 Shapley Value\n'
    f'원고: {pat_list[QUERY_S]}  →  피고: {pat_list[QUERY_D]}', fontsize=12)

for ax, sv, node_lbl in [
    (ax1, shap_src, f'원고 {str(pat_list[QUERY_S])[:15]}'),
    (ax2, shap_dst, f'피고 {str(pat_list[QUERY_D])[:15]}')
]:
    slist = sorted(sv.items(), key=lambda x: -abs(x[1]))
    gnames = [g[0] for g in slist]; gvals = [g[1] for g in slist]
    ax.barh(range(len(gnames)), gvals,
            color=['green' if v > 0 else 'red' for v in gvals], alpha=0.8)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(range(len(gnames))); ax.set_yticklabels(gnames, fontsize=8)
    ax.set_xlabel('SHAP Value (기여도)'); ax.set_title(node_lbl)

plt.tight_layout()
plt.savefig('xai_m7_graphshap.png', dpi=120, bbox_inches='tight')
plt.show(); print("저장: xai_m7_graphshap.png")


## Summary: 7가지 XAI 결과 비교


In [ ]:
print("="*65)
print(f"분석 대상:  {pat_list[QUERY_S]}  →  {pat_list[QUERY_D]}")
print(f"예측 확률:  {QUERY_P:.4f}  (threshold: {best_thr:.2f})")
print(f"예측 결과:  {'소송 예측 O' if QUERY_P >= best_thr else '미예측 X'}")
print("="*65)

print("\n[저장된 시각화 파일]")
xai_files = [
    ('M1 Attention Visualization', 'xai_m1_attention.png',   '어떤 이웃 노드가 임베딩에 기여했는지'),
    ('M2 Grad-CAM',                'xai_m2_gradcam.png',     '어떤 레이어/피처가 예측에 기여했는지'),
    ('M3 Counterfactual',          'xai_m3_counterfactual.png','엣지 제거시 가장 큰 확률 변화'),
    ('M4 GNNExplainer',            'xai_m4_gnnexplainer.png', '예측 유지에 필요한 핵심 엣지'),
    ('M5 PGExplainer',             'xai_m5_pgexplainer.png',  '이웃 중 소송 예측에 기여하는 특허'),
    ('M6 SubgraphX',               'xai_m6_subgraphx.png',   'Shapley 기준 핵심 이웃 노드'),
    ('M7 GraphSHAP',               'xai_m7_graphshap.png',   'struct vs FAISS 피처 그룹 기여도'),
]
for name, fname, desc in xai_files:
    print(f"  [{name}] {fname}")
    print(f"    → {desc}")
    print()

print("\n[전체 test 예측 결과 (상위 15개)]")
print(f"{'#':<4} {'원고':^22} {'피고':^22} {'P(sue)':>8} {'예측':>5}")
print('-'*65)
for k_i, (s, d, p) in enumerate(test_pairs[:15]):
    flag = 'O' if p >= best_thr else '-'
    print(f"{k_i+1:<4} {str(pat_list[s])[:22]:<22} {str(pat_list[d])[:22]:<22} {p:>7.3f} {flag:>5}")
